## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark
### Датасет: Steam Games Dataset

### Часть 1: Загрузка и очистка данных

В данной части работы рассмотрены:
* загрузка данных из CSV файла;
* базовые преобразования данных;
* приведение типов данных;
* сохранение данных в формате Parquet.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    col,
    when,
    split,
    trim,
    size,
    isnan,
    isnull,
    count,
    sum,
    lit,
    explode, 
    collect_list,
    filter as spark_filter
)
from pyspark.sql.types import (
    StructType, StructField, 
    StringType, IntegerType, FloatType, BooleanType, ArrayType
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    conf = SparkConf()
    conf.setAppName("Steam Games Analysis")
    conf.setMaster("local[2]")  # Используем 2 ядра для экономии памяти
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.executor.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.executor.cores", "2")
    conf.set("spark.sql.shuffle.partitions", "4")
    conf.set("spark.sql.adaptive.enabled", "false")
    conf.set("spark.memory.fraction", "0.6")
    conf.set("spark.memory.storageFraction", "0.3")
    conf.set("spark.ui.enabled", "false")
    
    return conf

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

### Загрузка данных Steam Games

Датасет Steam Games доступен по адресу: https://www.kaggle.com/datasets/mexwell/steam-games-dataset

Описание датасета:
- 136,000+ игр из Steam
- Содержит метаданные, цены, разработчиков, издателей, жанры
- Данные обновляются ежедневно

**Важные столбцы:**
- `app_id` – Уникальный идентификатор приложения Steam
- `name` – Название игры
- `release_date` – Дата релиза
- `price` – Текущая цена в USD
- `estimated_owners` – Оценочное количество владельцев
- `developers` – Список разработчиков
- `publishers` – Список издателей
- `genres` – Список жанров
- `positive` / `negative` – Количество положительных/отрицательных отзывов
- `recommendations` – Количество рекомендаций
- `average_playtime_forever` – Среднее время игры (в минутах)

In [5]:
# Путь к файлу с данными
path = "data/steam_games.csv"

# Загружаем данные
df = (spark.read.format("csv")
      .option("header", "true")
      .option("escape", "\"")
      .option("multiLine", "true")
      .option("quote", "\"")
      .load(path)
)

print(" Данные загружены")
print(f"Количество записей: {df.count()}")

 Данные загружены
Количество записей: 139697


In [6]:
# Показываем структуру данных
df.printSchema()

root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- price_status: string (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- developers: string (nullable = true)
 |-- publishers: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- recommendations: string (nullable = true)
 |-- peak_ccu: string (nullable = true)
 |-- metacritic_score: string (nullable = true)
 |-- user_score: string (nullable = true)
 |-- average_playtime_forever: string (nullable = true)
 |-- median_playtime_forever: string (nullable = true)
 |-- average_playtime_2weeks: string (nullable = true)
 |-- median_playtime_2weeks: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- about_the_game: s

In [7]:
# Показываем первые строки
df.show(5, truncate=50)

+------+------------------------------------------+------------+-----+------------+----------------+--------------------------------+--------------------------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+--------+--------+---------------+--------+----------------+----------+------------------------+-----------------------+-----------------------+----------------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------------+---------+--------------------------------------------------+--------------------------------------------------+--------------------+-------+-----+-----+--------------------------------------------------+--------------------------------------------------+------+------------------------------+----

### Выбор и преобразование столбцов

Для анализа оставим наиболее важные столбцы:

In [8]:
# Выбираем необходимые столбцы
df_selected = df.select(
    "app_id",
    "name",
    "release_date",
    "price",
    "estimated_owners",
    "developers",
    "publishers",
    "genres",
    "categories",
    "positive",
    "negative",
    "recommendations",
    "average_playtime_forever",
    "steam_store_available",
    "steam_spy_available"
)

print("Столбцы выбраны")
df_selected.show(5)

Столбцы выбраны
+------+--------------------+------------+-----+----------------+--------------------+--------------------+--------------------+--------------------+--------+--------+---------------+------------------------+---------------------+-------------------+
|app_id|                name|release_date|price|estimated_owners|          developers|          publishers|              genres|          categories|positive|negative|recommendations|average_playtime_forever|steam_store_available|steam_spy_available|
+------+--------------------+------------+-----+----------------+--------------------+--------------------+--------------------+--------------------+--------+--------+---------------+------------------------+---------------------+-------------------+
|881500|    Steamulator 2019|  2019-01-30| 0.99|       0 - 20000|["Mutant Entertai...|["Mutant Entertai...|      ["Simulation"]|["Single-player",...|      11|      14|            0.0|                       0|                 True| 

### Преобразование типов данных

In [9]:
def transform_steam_data(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    # Преобразование числовых столбцов
    data = data.withColumn("price", col("price").cast(FloatType()))
    data = data.withColumn("positive", col("positive").cast(IntegerType()))
    data = data.withColumn("negative", col("negative").cast(IntegerType()))
    data = data.withColumn("recommendations", col("recommendations").cast(IntegerType()))
    data = data.withColumn("average_playtime_forever", col("average_playtime_forever").cast(FloatType()))
    
    # Преобразование булевых столбцов
    data = data.withColumn("steam_store_available", 
                          when(col("steam_store_available") == "True", True)
                          .when(col("steam_store_available") == "False", False)
                          .cast(BooleanType()))
    data = data.withColumn("steam_spy_available",
                          when(col("steam_spy_available") == "True", True)
                          .when(col("steam_spy_available") == "False", False)
                          .cast(BooleanType()))
    
    # Извлечение года из даты релиза
    data = data.withColumn("release_year", 
                          regexp_replace(col("release_date"), r"-.*$", "").cast(IntegerType()))
    
    # Преобразование списков
    data = data.withColumn("developers", split(regexp_replace(col("developers"), r"^\[|\]$", ""), r",\s*"))
    data = data.withColumn("publishers", split(regexp_replace(col("publishers"), r"^\[|\]$", ""), r",\s*"))
    data = data.withColumn("genres", split(regexp_replace(col("genres"), r"^\[|\]$", ""), r",\s*"))
    data = data.withColumn("categories", split(regexp_replace(col("categories"), r"^\[|\]$", ""), r",\s*"))
    
    # Количество жанров и категорий
    data = data.withColumn("num_genres", size(col("genres")))
    data = data.withColumn("num_categories", size(col("categories")))
    
    return data

In [10]:
# Применяем преобразования
df_transformed = transform_steam_data(df_selected)
print("Преобразования применены")

Преобразования применены


In [11]:
# Проверяем результат
df_transformed.printSchema()

root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: float (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- developers: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- publishers: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- positive: integer (nullable = true)
 |-- negative: integer (nullable = true)
 |-- recommendations: integer (nullable = true)
 |-- average_playtime_forever: float (nullable = true)
 |-- steam_store_available: boolean (nullable = true)
 |-- steam_spy_available: boolean (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- num_genres: integer (nullable = false)
 |-- num_categories: integer (nullable = false)



In [12]:
df_transformed.show(5, truncate=50)

+------+------------------------------------------+------------+-----+----------------+--------------------------------+--------------------------------+--------------------------------------------------+--------------------------------------------------+--------+--------+---------------+------------------------+---------------------+-------------------+------------+----------+--------------+
|app_id|                                      name|release_date|price|estimated_owners|                      developers|                      publishers|                                            genres|                                        categories|positive|negative|recommendations|average_playtime_forever|steam_store_available|steam_spy_available|release_year|num_genres|num_categories|
+------+------------------------------------------+------------+-----+----------------+--------------------------------+--------------------------------+--------------------------------------------------+----

### Статистика пропущенных значений

In [13]:
def calculate_missing_stats(data: DataFrame) -> DataFrame:
    """
    Рассчитывает количество пропущенных значений для каждого столбца.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: DataFrame со статистикой пропущенных значений.
    """
    total_rows = data.count()
    
    missing_stats = []
    for column in data.columns:
        null_count = data.filter(col(column).isNull()).count()
        missing_stats.append((column, null_count, null_count/total_rows))
    
    return spark.createDataFrame(missing_stats, 
                                 ["column", "missing_count", "missing_ratio"])

missing_stats = calculate_missing_stats(df_transformed)
missing_stats.show(truncate=False)

+------------------------+-------------+---------------------+
|column                  |missing_count|missing_ratio        |
+------------------------+-------------+---------------------+
|app_id                  |0            |0.0                  |
|name                    |16           |1.1453359771505472E-4|
|release_date            |83           |5.941430381468464E-4 |
|price                   |1174         |0.008403902732342141 |
|estimated_owners        |0            |0.0                  |
|developers              |0            |0.0                  |
|publishers              |0            |0.0                  |
|genres                  |0            |0.0                  |
|categories              |0            |0.0                  |
|positive                |0            |0.0                  |
|negative                |0            |0.0                  |
|recommendations         |31892        |0.22829409364553283  |
|average_playtime_forever|0            |0.0            

### Заполнение пропущенных значений

In [14]:
def fill_missing_values(data: DataFrame) -> DataFrame:
    """
    Заполняет пропущенные значения в DataFrame.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: DataFrame с заполненными пропущенными значениями.
    """
    # Заполняем числовые столбцы средними значениями
    numeric_cols = ["price", "positive", "negative", "recommendations", 
                    "average_playtime_forever", "num_genres", "num_categories"]
    for col_name in numeric_cols:
        if col_name in data.columns:
            avg_val = data.select(col(col_name)).filter(col(col_name).isNotNull()).agg({col_name: "avg"}).collect()[0][0]
            if avg_val is not None:
                data = data.withColumn(col_name, when(col(col_name).isNull(), avg_val).otherwise(col(col_name)))
    
    # Заполняем строковые столбцы пустой строкой или "Unknown"
    string_cols = ["name", "release_date", "estimated_owners"]
    for col_name in string_cols:
        if col_name in data.columns:
            data = data.withColumn(col_name, when(col(col_name).isNull(), "Unknown").otherwise(col(col_name)))
    
    # Заполняем списки пустыми массивами
    array_cols = ["developers", "publishers", "genres", "categories"]
    for col_name in array_cols:
        if col_name in data.columns:
            data = data.withColumn(col_name, when(col(col_name).isNull(), split(lit(""), ",")).otherwise(col(col_name)))
    
    # Заполняем булевы столбцы значением False
    bool_cols = ["steam_store_available", "steam_spy_available"]
    for col_name in bool_cols:
        if col_name in data.columns:
            data = data.withColumn(col_name, when(col(col_name).isNull(), False).otherwise(col(col_name)))
    
    return data

In [15]:
# Заполняем пропущенные значения
df_cleaned = fill_missing_values(df_transformed)
print("Пропущенные значения заполнены")
df_cleaned = df_cleaned.filter(col("release_year").isNotNull())
# Проверяем, что пропусков больше нет
missing_stats_after = calculate_missing_stats(df_cleaned)
missing_stats_after.filter(col("missing_count") > 0).show()

Пропущенные значения заполнены
+------+-------------+-------------+
|column|missing_count|missing_ratio|
+------+-------------+-------------+
+------+-------------+-------------+



### Сохранение данных в формате Parquet

In [16]:
def clean_array_column(data: DataFrame, column_name: str) -> DataFrame:
    """
    Очищает массив строк от мусора: фигурных скобок, кавычек,
    префиксов вида id:, description: и чисто числовых значений.
    """
    other_cols = [c for c in data.columns if c != column_name]
    
    # Разворачиваем массив
    exploded = data.select(*other_cols, explode(col(column_name)).alias("element"))
    
    # Очищаем каждый элемент
    exploded = exploded.withColumn(
        "element",
        trim(
            regexp_replace(
                regexp_replace(col("element"), r'[{}\[\]"]', ''),
                r'(id|description|name)\s*:\s*', ''
            )
        )
    )
    
    # Убираем пустые, null и чисто числовые значения
    exploded = exploded.filter(
        (col("element") != "") 
        & (col("element").isNotNull())
        & (~col("element").rlike(r'^\d+$'))
    )
    
    # Собираем обратно в массив
    restored = exploded.groupBy(*other_cols).agg(
        collect_list("element").alias(column_name)
    )
    return restored

# Применяем ко всем массивам
for arr_col in ["genres", "developers", "publishers", "categories"]:
    df_cleaned = clean_array_column(df_cleaned, arr_col)
    print(f" Очищен столбец: {arr_col}")

 Очищен столбец: genres
 Очищен столбец: developers
 Очищен столбец: publishers
 Очищен столбец: categories


In [17]:
# Сохраняем очищенные данные в Parquet
df_cleaned.write.mode("overwrite").parquet("steam_games_cleaned.parquet")
print("Данные сохранены в Parquet: steam_games_cleaned.parquet")

# Для проверки читаем сохраненный файл
df_check = spark.read.parquet("steam_games_cleaned.parquet")
print(f"Количество записей в сохраненном файле: {df_check.count()}")
df_check.show(3)

Данные сохранены в Parquet: steam_games_cleaned.parquet
Количество записей в сохраненном файле: 129587
+-------+--------------------+------------+------------------+----------------+--------+--------+---------------+------------------------+---------------------+-------------------+------------+----------+--------------+--------------------+------------------+------------------+--------------------+
| app_id|                name|release_date|             price|estimated_owners|positive|negative|recommendations|average_playtime_forever|steam_store_available|steam_spy_available|release_year|num_genres|num_categories|              genres|        developers|        publishers|          categories|
+-------+--------------------+------------+------------------+----------------+--------+--------+---------------+------------------------+---------------------+-------------------+------------+----------+--------------+--------------------+------------------+------------------+-------------------

In [18]:
# Останавливаем сессию Spark
spark.stop()
print("Сессия Spark остановлена")

Сессия Spark остановлена
